# Montage diff debugger (contacts & pairs)

Same shape as `events_debug.ipynb`, but for the `MontagePipeline` outputs.

Part 1 (`df_{contacts,pairs}_summary_all.csv`): one-hot view of which
(subject, session, space) rows have which columns differing.

Part 2: deep-dive on a single (sub, exp, sess, space, acquisition). We
bypass the pipeline's outer space-loop and run the comparator directly,
since `MontagePipeline.run()` only returns the *last* space's result.

In [15]:
import ast
from pathlib import Path

import pandas as pd

RESULTS_DIR = Path('/home1/zrentala/eeg-validation/results')
EXPERIMENT_RESULT = 'FR1_final_neurorad'  # change to FR1_final, catFR1_final_pyedflib, etc.
ACQUISITION = 'pairs'                   # 'contacts' or 'pairs'

SUMMARY_CSV = RESULTS_DIR / EXPERIMENT_RESULT / f'df_{ACQUISITION}_summary_all.csv'
COLUMN_SUMMARY_CSV = RESULTS_DIR / EXPERIMENT_RESULT / f'df_{ACQUISITION}_column_summary_all.csv'
MISMATCHES_CSV = RESULTS_DIR / EXPERIMENT_RESULT / f'df_{ACQUISITION}_mismatches_all.csv'
print(SUMMARY_CSV)

/home1/zrentala/eeg-validation/results/FR1_final_neurorad/df_pairs_summary_all.csv


In [16]:
def _parse_list(val):
    if isinstance(val, list):
        return val
    if pd.isna(val) or val in ('', '[]'):
        return []
    return ast.literal_eval(val)


df = pd.read_csv(SUMMARY_CSV)
df['differing_columns'] = df['differing_columns'].apply(_parse_list)
broken = df[df['differing_columns'].map(len) > 0].copy()

all_cols = sorted({c for cols in broken['differing_columns'] for c in cols})
onehot = pd.DataFrame(0, index=broken.index, columns=all_cols, dtype=int)
for idx, cols in broken['differing_columns'].items():
    onehot.loc[idx, cols] = 1

id_cols = ['subject', 'experiment', 'session', 'space']
onehot = pd.concat([broken[id_cols].reset_index(drop=True),
                    onehot.reset_index(drop=True)], axis=1)
print(f'{len(onehot)} (subject, session, space) rows with differing columns; '
      f'{len(all_cols)} unique columns')
onehot

286 (subject, session, space) rows with differing columns; 27 unique columns


,subject,experiment,session,space,avg.corrected.x,avg.corrected.y,avg.corrected.z,avg.x,avg.y,avg.z,...,mni.y,mni.z,tal.x,tal.y,tal.z,type_1,type_2,vox.x,vox.y,vox.z
0,R1020J,FR1,NaN,MNI152NLin6ASym,0,0,0,0,0,0,...,1,1,0,0,0,0,0,0,0,0
1,R1052E,FR1,NaN,MNI152NLin6ASym,0,0,0,0,0,0,...,1,1,0,0,0,0,0,0,0,0
2,R1052E,FR1,1.0,MNI152NLin6ASym,0,0,0,0,0,0,...,1,1,0,0,0,0,0,0,0,0
3,R1059J,FR1,NaN,MNI152NLin6ASym,0,0,0,0,0,0,...,1,1,0,0,0,0,0,0,0,0
4,R1059J,FR1,1.0,MNI152NLin6ASym,0,0,0,0,0,0,...,1,1,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
281,R1467M,FR1,NaN,Pixels,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,1,1
282,R1542J,FR1,NaN,Pixels,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,1,1
283,R1542J,FR1,1.0,Pixels,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,1,1
284,R1542J,FR1,2.0,Pixels,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,1,1


In [17]:
# How often each column differs.
onehot[all_cols].sum().sort_values(ascending=False).to_frame('n_rows')

,n_rows
type_2,165
type_1,165
vox.z,137
vox.x,137
vox.y,135
label,51
avg.x,27
avg.y,27
avg.z,27
mni.z,23


In [18]:
# How differing rows are distributed across spaces.
onehot.groupby('space').size().sort_values(ascending=False).to_frame('n_rows_with_diffs')

,n_rows_with_diffs
space,
Pixels,142
MNI152NLin6ASym,36
fsaverage,27
fsnative,27
fsaverageBrainshift,22
fsnativeBrainshift,22
Talairach,5
fsnativeDural,5


In [19]:
# Example: rows where 'wb.region' differs (or whichever column you care about).
TARGET = 'wb.region'
onehot[onehot[TARGET] == 1] if TARGET in onehot.columns else f'no rows with {TARGET} differing'

'no rows with wb.region differing'

In [20]:
# Per-session, per-space MontagePipeline status — useful when summary CSV is
# missing rows you expected (skip reasons live in df_montage_status_*.csv).
status_files = sorted((RESULTS_DIR / EXPERIMENT_RESULT).glob(f'df_montage_status_{ACQUISITION}*.csv'))
if status_files:
    status = pd.concat([pd.read_csv(p) for p in status_files], ignore_index=True)
    skipped = status[status['skipped'] == True] if 'skipped' in status.columns else status
    print(f'{len(status)} status rows, {len(skipped)} skipped')
    skipped['reason'].value_counts() if len(skipped) else 'no skips'
else:
    print('no per-session df_montage_status_* files found')

no per-session df_montage_status_* files found


## Deep-dive: one (sub, exp, sess, space)

Run the loaders + preparers + comparator directly — same machinery as
`MontagePipeline` but for a single chosen space, so we can inspect the
aligned CML/BIDS frames and per-row mismatches.

In [29]:
import sys
sys.path.insert(0, '/home1/zrentala/eeg-validation')

from eeg_validation.loaders.cml import load_cml_contacts_and_pairs
from eeg_validation.loaders.bids import get_reader, load_bids_channels
from eeg_validation.preparers.montage import prep_contacts, prep_pairs, BIDS_TO_CML_SPACE
from eeg_validation.comparators.dataframe import DataFrameComparator

SUB, EXP, SESS = 'R1137E', 'FR1', 1
LOC, MON = 0, 0
BIDS_ROOT = '/data/LTP_BIDS/pyedflib/FR1/'   # adjust per cohort
TARGET_SPACE = 'Pixels'             # any value from onehot['space'].unique()
ATOL_COORDS = 1e-3
RTOL_COORDS = 0.0

cml_key = BIDS_TO_CML_SPACE.get(TARGET_SPACE)
print(f'BIDS space {TARGET_SPACE!r} -> CML key {cml_key!r}')

BIDS space 'Pixels' -> CML key 'vox'


In [28]:
import math
import json
base = Path('/protocols/r1/subjects')
proc = base / SUB / 'localizations' / str(LOC) / 'montages' / str(MON) / 'neuroradiology' / 'current_processed'
pairs    = json.load(open(proc / 'pairs.json'))
contacts = json.load(open(proc / 'contacts.json'))
sub = list(pairs)[0]
tol = 1e-6
diffs = []
for label, p in pairs[sub]['pairs'].items():
    c1, c2 = label.split('-')
    for atlas, coords in p['atlases'].items():
        x_p, y_p, z_p = coords.get('x'), coords.get('y'), coords.get('z')
        c1_a = contacts[sub]['contacts'].get(c1, {}).get('atlases', {}).get(atlas, {})
        c2_a = contacts[sub]['contacts'].get(c2, {}).get('atlases', {}).get(atlas, {})
        vals = [x_p, y_p, z_p,
                c1_a.get('x'), c1_a.get('y'), c1_a.get('z'),
                c2_a.get('x'), c2_a.get('y'), c2_a.get('z')]
        if any(v is None for v in vals):
            continue
        for axis, vp in zip('xyz', (x_p, y_p, z_p)):
            mid = (c1_a[axis] + c2_a[axis]) / 2
            if not math.isclose(vp, mid, abs_tol=tol):
                diffs.append((label, atlas, axis, vp, mid, vp - mid))

print(f'{len(diffs)} (pair, atlas, axis) triples with |pair - midpoint| > {tol}')
import pandas as pd
pd.DataFrame(diffs, columns=['pair', 'atlas', 'axis', 'pairs.json', 'midpoint(contacts.json)', 'delta']) \
  .sort_values('delta', key=lambda s: s.abs(), ascending=False).head(30)

261 (pair, atlas, axis) triples with |pair - midpoint| > 1e-06


,pair,atlas,axis,pairs.json,midpoint(contacts.json),delta
257,RSTC6-RSTC7,mni,z,-30.631677,-29.105063,-1.526613
112,RFD3-RFD4,mni,y,-10.585900,-9.074275,-1.511625
230,RSTB4-RSTB5,mni,z,-30.433344,-31.906765,1.473420
115,RFD4-RFD5,mni,y,4.062490,2.657120,1.405370
105,RFD1-RFD2,mni,x,18.496500,19.871950,-1.375450
229,RSTB4-RSTB5,mni,y,-24.464900,-25.449650,0.984750
111,RFD3-RFD4,mni,x,34.297000,33.352550,0.944450
107,RFD1-RFD2,mni,z,78.570009,79.499524,-0.929516
123,RFD7-RFD8,mni,x,55.911300,55.024850,0.886450
241,RSTC1-RSTC2,mni,y,-71.772300,-72.656200,0.883900


In [39]:
contacts_cml, pairs_cml = load_cml_contacts_and_pairs(SUB, EXP, SESS, LOC, MON)
print(f'CML contacts: {len(contacts_cml)} rows | CML pairs: '
      f'{len(pairs_cml) if pairs_cml is not None else 0} rows')

reader = get_reader(SUB, EXP, SESS, BIDS_ROOT)
elec = reader.load_electrodes(space=TARGET_SPACE)
channels = reader.load_channels(acquisition="bipolar")
combined_channels = reader.load_combined_channels(acquisition="bipolar", space=TARGET_SPACE)
print(f'BIDS electrodes (space={TARGET_SPACE}): {len(elec)} rows')

if ACQUISITION == 'pairs':
    ch_bip = load_bids_channels(reader, acquisition='bipolar')
    ch_bip = ch_bip[ch_bip['name'].astype(str).str.contains('-')]
    df_bids = prep_pairs(elec, ch_bip, cml_key=cml_key)
    df_cml = pairs_cml
    print(f'BIDS bipolar channels: {len(ch_bip)} | prepped pairs: {len(df_bids)}')
else:
    df_bids = prep_contacts(elec, cml_key=cml_key)
    df_cml = contacts_cml
    print(f'prepped contacts: {len(df_bids)}')

CML contacts: 80 rows | CML pairs: 71 rows
BIDS electrodes (space=Pixels): 80 rows
BIDS bipolar channels: 71 | prepped pairs: 71


/usr/global/ubuntu/miniforge3/25.3.1/envs/workshop_311_rhino2b/lib/python3.11/site-packages/cmlreaders/readers/electrodes.py:241: MissingCoordinatesWarning: Could not load MNI coordinates
  warnings.warn(cmlreaders.warnings.MissingCoordinatesWarning(


In [31]:
pairs_cml

,contact_1,contact_2,label,id,is_explicit,is_stim_only,type_1,type_2,avg.region,avg.x,...,stein.y,stein.z,tal.region,tal.x,tal.y,tal.z,wb.region,wb.x,wb.y,wb.z
0,1,2,L1TPODL1-L1TPODL2,l1tpodl.1-l1tpodl.2,False,False,D,D,temporalpole,-28.995,...,NaN,NaN,None,-28.84780,7.175805,-33.330000,Left Cerebral White Matter,NaN,NaN,NaN
1,2,3,L1TPODL2-L1TPODL3,l1tpodl.2-l1tpodl.3,False,False,D,D,temporalpole,-33.070,...,NaN,NaN,None,-32.93905,4.331205,-31.906550,Left Cerebral White Matter,NaN,NaN,NaN
2,3,4,L1TPODL3-L1TPODL4,l1tpodl.3-l1tpodl.4,False,False,D,D,inferiortemporal,-36.865,...,NaN,NaN,None,-36.75230,1.018873,-30.555600,Left Cerebral White Matter,NaN,NaN,NaN
3,4,5,L1TPODL4-L1TPODL5,l1tpodl.4-l1tpodl.5,False,False,D,D,inferiortemporal,-40.675,...,NaN,NaN,None,-40.58015,-2.050171,-29.166250,Left Cerebral White Matter,NaN,NaN,NaN
4,5,6,L1TPODL5-L1TPODL6,l1tpodl.5-l1tpodl.6,False,False,D,D,middletemporal,-43.985,...,NaN,NaN,None,-43.90915,-5.066250,-28.081600,Left Cerebral White Matter,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
66,75,76,L15OF2D5-L15OF2D6,l15of2d.5-l15of2d.6,False,False,D,D,lateralorbitofrontal,-33.520,...,NaN,NaN,None,-33.26040,31.998600,-3.845585,Left Cerebral White Matter,NaN,NaN,NaN
67,76,77,L15OF2D6-L15OF2D7,l15of2d.6-l15of2d.7,False,False,D,D,parstriangularis,-33.200,...,NaN,NaN,None,-32.92585,34.264850,1.153260,Left Cerebral White Matter,NaN,NaN,NaN
68,77,78,L15OF2D7-L15OF2D8,l15of2d.7-l15of2d.8,False,False,D,D,parstriangularis,-32.870,...,NaN,NaN,None,-32.58760,36.584850,5.365065,Left Cerebral White Matter,NaN,NaN,NaN
69,78,79,L15OF2D8-L15OF2D9,l15of2d.8-l15of2d.9,False,False,D,D,rostralmiddlefrontal,-31.815,...,NaN,NaN,None,-31.51995,39.177000,10.319805,Left Cerebral White Matter,NaN,NaN,NaN


In [40]:
channels

,name,type,units,low_cutoff,high_cutoff,reference,group,sampling_frequency,description,notch,status,status_description
0,L1TPOD1-L1TPOD2,SEEG,uV,NaN,NaN,bipolar,LTPODL,500,depth,NaN,good,NaN
1,L1TPOD2-L1TPOD3,SEEG,uV,NaN,NaN,bipolar,LTPODL,500,depth,NaN,good,NaN
2,L1TPOD3-L1TPOD4,SEEG,uV,NaN,NaN,bipolar,LTPODL,500,depth,NaN,good,NaN
3,L1TPOD4-L1TPOD5,SEEG,uV,NaN,NaN,bipolar,LTPODL,500,depth,NaN,good,NaN
4,L1TPOD5-L1TPOD6,SEEG,uV,NaN,NaN,bipolar,LTPODL,500,depth,NaN,good,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
66,L15OF25-L15OF26,SEEG,uV,NaN,NaN,bipolar,LOFD,500,depth,NaN,good,NaN
67,L15OF26-L15OF27,SEEG,uV,NaN,NaN,bipolar,LOFD,500,depth,NaN,good,NaN
68,L15OF27-L15OF28,SEEG,uV,NaN,NaN,bipolar,LOFD,500,depth,NaN,good,NaN
69,L15OF28-L15OF29,SEEG,uV,NaN,NaN,bipolar,LOFD,500,depth,NaN,good,NaN


In [37]:
combined_channels

,name,type,units,low_cutoff,high_cutoff,reference,group,sampling_frequency,description,notch,...,z_ch1,wb.region_ch2,ind.region_ch2,stein.region_ch2,x_ch2,y_ch2,z_ch2,x_mid,y_mid,z_mid
0,L1TPOD1-L1TPOD2,SEEG,uV,NaN,NaN,bipolar,LTPODL,500,depth,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,L1TPOD2-L1TPOD3,SEEG,uV,NaN,NaN,bipolar,LTPODL,500,depth,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,L1TPOD3-L1TPOD4,SEEG,uV,NaN,NaN,bipolar,LTPODL,500,depth,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,L1TPOD4-L1TPOD5,SEEG,uV,NaN,NaN,bipolar,LTPODL,500,depth,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,L1TPOD5-L1TPOD6,SEEG,uV,NaN,NaN,bipolar,LTPODL,500,depth,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
66,L15OF25-L15OF26,SEEG,uV,NaN,NaN,bipolar,LOFD,500,depth,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
67,L15OF26-L15OF27,SEEG,uV,NaN,NaN,bipolar,LOFD,500,depth,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
68,L15OF27-L15OF28,SEEG,uV,NaN,NaN,bipolar,LOFD,500,depth,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
69,L15OF28-L15OF29,SEEG,uV,NaN,NaN,bipolar,LOFD,500,depth,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [32]:
elec

,name,x,y,z,size,group,hemisphere,type,wb.region,ind.region,das.region,stein.region
0,L1TPODL1,315,217,138,-999,L1TPODL,R,depth,Left TMP temporal pole,temporalpole,NaN,NaN
1,L1TPODL2,324,221,141,-999,L1TPODL,R,depth,Left Cerebral White Matter,entorhinal,NaN,NaN
2,L1TPODL3,332,227,144,-999,L1TPODL,R,depth,Left Cerebral White Matter,inferiortemporal,NaN,NaN
3,L1TPODL4,340,233,147,-999,L1TPODL,R,depth,Left Cerebral White Matter,inferiortemporal,NaN,NaN
4,L1TPODL5,348,238,150,-999,L1TPODL,R,depth,Left Cerebral White Matter,inferiortemporal,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
75,L15OF2D6,324,162,184,-999,L15OF2D,R,depth,Left Cerebral White Matter,parstriangularis,NaN,NaN
76,L15OF2D7,324,158,190,-999,L15OF2D,R,depth,Left Cerebral White Matter,parstriangularis,NaN,NaN
77,L15OF2D8,322,152,195,-999,L15OF2D,R,depth,Left Cerebral White Matter,rostralmiddlefrontal,NaN,NaN
78,L15OF2D9,319,147,203,-999,L15OF2D,R,depth,Left Cerebral White Matter,rostralmiddlefrontal,NaN,NaN


In [ ]:
comparator = DataFrameComparator(
    tolerant_numeric=True, rtol=RTOL_COORDS, atol=ATOL_COORDS, sort_keys=['label'],
)
result = comparator.compare(
    df_cml, df_bids,
    label_a='CML', label_b='BIDS', space=TARGET_SPACE,
    subject=SUB, experiment=EXP, session=SESS, return_aligned=True,
)
print('ok =', result.ok)

In [ ]:
# Session-level summary (differing columns, only-in-A, only-in-B, row counts).
result.df_summary.T

In [ ]:
# Per-column mismatch counts (only columns with n_mismatches > 0 matter).
result.df_detail[result.df_detail['n_mismatches'] > 0]

In [ ]:
# The actual differing values, row by row (up to max_mismatches=20 per column).
# Columns: subject, experiment, session, column, i (row index into aligned
# frames), CML (value in aligned_a), BIDS (value in aligned_b).
result.df_mismatches

In [ ]:
# Side-by-side view: for every row index that has ANY differing column,
# show the CML and BIDS values of the differing columns next to each other.
a, b = result.aligned_a, result.aligned_b
diff_cols = result.df_summary.iloc[0]['differing_columns']
bad_idx = sorted(result.df_mismatches['i'].unique().tolist()) if len(result.df_mismatches) else []

if not bad_idx:
    print('no differing rows')
else:
    pieces = []
    for col in diff_cols:
        pieces.append(pd.DataFrame({
            f'{col}__CML': a[col].iloc[bad_idx].values,
            f'{col}__BIDS': b[col].iloc[bad_idx].values,
        }, index=bad_idx))
    side_by_side = pd.concat(pieces, axis=1)
    side_by_side.index.name = 'row_i'
    print(f'{len(bad_idx)} rows with at least one differing column')
    side_by_side

In [ ]:
# Show the full aligned rows (all shared columns) for the mismatching
# indices, with CML and BIDS interleaved so you can see the surrounding
# context (label, type/group, neighbouring coords).
if bad_idx:
    rows_cml = a.iloc[bad_idx].reset_index(drop=True).add_suffix('__CML')
    rows_bids = b.iloc[bad_idx].reset_index(drop=True).add_suffix('__BIDS')
    interleaved = pd.concat([rows_cml, rows_bids], axis=1)
    interleaved.insert(0, 'row_i', bad_idx)
    interleaved

### Structural diagnostics: rows missing on one side

If `n_rows_a != n_rows_b`, the comparator alignment will leave rows with
mostly-NaN values — the underlying issue is one side has labels the
other doesn't. Outer-join on `label` to find those, since label is the
comparator's sort key.

In [ ]:
key = 'label'
if key not in df_cml.columns or key not in df_bids.columns:
    print(f"'{key}' missing from one side — columns:")
    print('  CML :', sorted(df_cml.columns)[:20], '...')
    print('  BIDS:', sorted(df_bids.columns)[:20], '...')
else:
    cml_keys = df_cml[[key]].assign(_in_cml=1)
    bids_keys = df_bids[[key]].assign(_in_bids=1)
    outer = cml_keys.merge(bids_keys, on=key, how='outer', indicator=True)
    only_cml = outer[outer['_merge'] == 'left_only']
    only_bids = outer[outer['_merge'] == 'right_only']
    print(f'CML rows: {len(df_cml)}  |  BIDS rows: {len(df_bids)}')
    print(f'Only in CML: {len(only_cml)}  |  Only in BIDS: {len(only_bids)}')
    pd.concat([only_cml.assign(side='CML_only'), only_bids.assign(side='BIDS_only')])

## Global cross-reference (all sessions, this cohort)

Pre-computed `df_{contacts,pairs}_mismatches_all.csv` gives every per-row
diff across all sessions/spaces. Useful for spotting patterns without
re-running the pipeline.

In [ ]:
if MISMATCHES_CSV.exists():
    mm = pd.read_csv(MISMATCHES_CSV)
    print(f'{len(mm)} mismatch rows total')
    mm.head()
else:
    print(f'{MISMATCHES_CSV} not found')

In [ ]:
# Which columns drive most of the diffs across the cohort?
if MISMATCHES_CSV.exists():
    mm.groupby('column').size().sort_values(ascending=False).to_frame('n_mismatches')
else:
    print('skip — no mismatches CSV')

In [ ]:
# Same, broken out by space — different spaces fail in different ways.
if MISMATCHES_CSV.exists() and 'space' in mm.columns:
    mm.groupby(['column', 'space']).size().unstack('space', fill_value=0)
else:
    print('skip — no mismatches CSV or no space column')